# PLgrid client

Wysyła prompty tłumaczeniowe do API PLGrid i zbiera odpowiedzi modelu. Wczytuje prompty z `prompts.tsv`, odpytuje wybrany model asynchronicznie (z ponawianiem i wznawianiem przerwanej pracy), zapisuje odpowiedzi do TSV, a na końcu weryfikuje kompletność tłumaczeń względem oryginalnych transkrypcji.

#### 1. Zależności

In [ ]:
%pip install --quiet openai pandas tqdm httpx nest_asyncio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import asyncio
import random
import nest_asyncio
import httpx
import pandas as pd
from openai import OpenAI, AsyncOpenAI, RateLimitError, InternalServerError, APIConnectionError
from tqdm.asyncio import tqdm_asyncio

#### 2. Ustawienia

In [ ]:
ENDPOINT_URL = "https://llmlab.plgrid.pl/api/v1/"
def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "plgrid")

Wyświetlenie modeli dostępnych na PLGrid

In [ ]:
client_tmp = OpenAI(base_url=ENDPOINT_URL, api_key=API_KEY)
for m in client_tmp.models.list().data:
    print(m.id)

CYFRAGOVPL/Llama-PLLuM-70B-chat-250801
CYFRAGOVPL/pllum-12b-nc-chat-250715
Qwen/QwQ-32B
Qwen/Qwen3-Coder-30B-A3B-Instruct
Qwen/Qwen3-Embedding-0.6B
Qwen/Qwen3-VL-8B-Instruct
Qwen/Qwen3.5-122B-A10B
Qwen/Qwen3.5-397B-A17B-FP8
Qwen/Qwen3.6-27B
Qwen/Qwen3.6-35B-A3B
google/gemma-4-31B
meta-llama/Llama-3.3-70B-Instruct
speakleash/Bielik-11B-v2.6-Instruct
speakleash/Bielik-11B-v3.0-Instruct
zai-org/GLM-4.7-Flash


In [ ]:
MODEL_NAME = "Qwen/QwQ-32B" # Nazwa modelu; musi się zgadzać z tym, co jest dostępne na endpointcie
MODEL_SHORT = "QwQ-32B" # Skrócona nazwa; używana do nazewnictwa plików itp.

N_SLUGS = 1 # Ile slugów (TED talków) ma być przetworzonych; None == wszystkie

LANG_FILTER = None # None dla wszystkich, "pl" tylko dla polskiego itp

ENDPOINT_URL = "https://llmlab.plgrid.pl/api/v1/"

def load_api_key(path, name):
    for line in open(path).read().strip().splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise KeyError(f"Klucz '{name}' nie znaleziony w {path}")

API_KEY = load_api_key("./api_key.txt", "plgrid")

PROMPTS_PATH = "./data/prompts.tsv"
OUTPUT_DIR = f"./data/responses/{MODEL_SHORT}" # Katalog na odpowiedzi
RESPONSES_PATH = f"{OUTPUT_DIR}/responses.tsv"


# Parametry dotyczące przetwarzania i zapytań do modelu
CONCURRENCY = 1
JITTER_MAX = 2.0
TIMEOUT = 555.0
MAX_OUTPUT_TOKENS = 8000

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

os.makedirs(OUTPUT_DIR, exist_ok=True)

#### 3. Ładowanie promptów

Wczytuje prompty, opcjonalnie zawęża je do wybranych języków i kilku slugów, a następnie wznawia pracę. Scala już pobrane odpowiedzi po id i odrzuca te zakończone wyjątkiem, żeby pobrać je ponownie.

In [ ]:
prompts_df = pd.read_csv(PROMPTS_PATH, sep="\t")

if LANG_FILTER is not None:
    langs = [LANG_FILTER] if isinstance(LANG_FILTER, str) else list(LANG_FILTER)
    prompts_df = prompts_df[prompts_df["target_lang"].isin(langs)]
    print(f"Filtr jezyka: {langs}")

if N_SLUGS is not None:
    keep = prompts_df["slug"].drop_duplicates().head(N_SLUGS).tolist()
    prompts_df = prompts_df[prompts_df["slug"].isin(keep)]
    print(f"Ograniczono do {N_SLUGS} slugow: {keep}")

prompts_df = prompts_df.reset_index(drop=True)

df = prompts_df.copy()
df["response"] = pd.NA

if os.path.exists(RESPONSES_PATH):
    prev = pd.read_csv(RESPONSES_PATH, sep="\t")
    prev.loc[prev["response"].astype(str).str.startswith("EXCEPTION:"), "response"] = pd.NA
    done_map = prev.dropna(subset=["response"]).drop_duplicates("id").set_index("id")["response"]
    df["response"] = df["id"].map(done_map)
    print(f"Wznowiono (merge po id): {df['response'].notna().sum()}/{len(df)} juz pobranych")
else:
    print(f"Start od poczatku: {len(df)} promptow do wyslania")

Ograniczono do 1 slugow: ['adam_grant_how_to_stop_languishing_and_start_finding_flow']
Wznowiono (merge po id): 6/6 juz pobranych


#### 4. Wysyłanie

Asynchronicznie wysyła oczekujące prompty z ograniczeniem współbieżności (semafor) i losowym opóźnieniem. Każda próba jest ponawiana przy błędach limitu/serwera/połączenia, a wynik zapisywany przyrostowo po każdej odpowiedzi.

In [ ]:
client = AsyncOpenAI(
    base_url=ENDPOINT_URL,
    api_key=API_KEY,
    http_client=httpx.AsyncClient(timeout=httpx.Timeout(TIMEOUT)),
)
sem = asyncio.Semaphore(CONCURRENCY)
RETRIES = 3

async def get_response(i):
    async with sem:
        await asyncio.sleep(random.uniform(0, JITTER_MAX))
        for attempt in range(RETRIES):
            try:
                res = await client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{"role": "user", "content": df.at[i, "prompt"]}],
                    max_tokens=MAX_OUTPUT_TOKENS,
                )
                df.at[i, "response"] = (res.choices[0].message.content or "").strip()
                df.to_csv(RESPONSES_PATH, sep="\t", index=False)
                return
            except (RateLimitError, InternalServerError, APIConnectionError) as e:
                wait = 15 * (attempt + 1)
                print(f"{df.at[i, 'id']}: {type(e).__name__}, czekam {wait}s i probuje ponownie")
                await asyncio.sleep(wait)
        print(f"{df.at[i, 'id']}: poddaje sie po {RETRIES} probach")

pending = df[df["response"].isna()].index.tolist()
print(f"Do wyslania: {len(pending)} (concurrency={CONCURRENCY})")

nest_asyncio.apply()
await tqdm_asyncio.gather(*[get_response(i) for i in pending])
print("Gotowe")

Do wyslania: 0 (concurrency=1)


0it [00:00, ?it/s]

Gotowe


#### 5. Wyniki

In [ ]:
for i in df[df["response"].notna()].index:
    print("=" * 80)
    print(f"{df.at[i, 'target_lang']}  |  {df.at[i, 'id']}")
    print("=" * 80)
    print(df.at[i, "response"][:2000])
    print()

zh  |  zh--adam_grant_how_to_stop_languishing_and_start_finding_flow
好的，我需要把这段TED演讲的英文字幕翻译成中文，并且严格按照用户给的规则来操作。首先，用户特别强调要完全翻译每一个cue block，不能漏掉任何一个，也不能合并或者省略。每个cue block的编号和时间戳必须保持原样，只翻译文本部分。同时要注意使用口语化的现代词汇，保持原文的语气和节奏，还要注意专有名词不翻译，舞台指示如（笑声）要翻译成中文，并且保留TED的英文不译。

首先，我得仔细检查每一个cue，确保翻译每个都完成。我从第一个开始：

1
00:00:00,413 --> 00:00:02,707
I know you all have long to-do lists,
翻译成：我知道大家都有 长长的待办事项列表，
这和用户提供的参考样例一致，没问题。

接下来第二个：
2
00:00:02,749 --> 00:00:07,503
but I hate wasting time so much that I have a to-don't list.
参考样例是“但是我讨厌浪费时间做这些 所以我有个“不办”事项列表。”这里的“to-don't list”翻译成“不办事项列表”是对的，所以直接照做。

然后每个cue都要逐句翻译。需要注意像“Doomscrolling”这样的网络用语，可能翻译成“末日刷屏”比较合适，参考之前的样例。还有“revenge bedtime procrastination”需要准确表达，可能得直译为“报复性熬夜拖延”，但要确保读者能理解。

还有专有名词如Corey Keyes和Mariah Carey要保留原名，而“National Treasure”作为节目名可能直接保留不译，或者用官方译名。假设“National Treasure”没有官方翻译，那么保留英文。

对于舞台提示，比如(Laughter)要翻译成（笑声），(Applause)是（掌声）。保持这些，并且不要漏掉。

需要特别注意中文标点的使用，比如句号用“。”而不是“.”，还有逗号的正确使用。此外，保持段落结构，每个cue换行，确保格式正确。

可能会遇到一些挑战，比如如何将口语化的英文表达转化为自然的中文，同时保持原意。

#### 6. Weryfikacja

Sprawdza kompletność odpowiedzi: liczy bloki napisów w wejściu i wyjściu. Odcina rozumowanie modeli w modelach rozumujących

In [ ]:
import re

TS_RE = re.compile(r"\d{2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*\d{2}:\d{2}:\d{2}[,.]\d{1,3}")
transcripts = pd.read_csv("./data/ted_talks_transcripts.tsv", sep="\t").set_index("slug")


def n_cues(srt):
    if not isinstance(srt, str):
        return 0
    n = 0
    for block in re.split(r"\n\s*\n", srt.strip()):
        bl = [l for l in block.split("\n") if l.strip()]
        p = next((j for j, l in enumerate(bl) if TS_RE.search(l)), None)
        if p is not None and [l for l in bl[p + 1:] if l.strip()]:
            n += 1
    return n


def strip_think(text):
    """Usuwa sekcję rozumowania modelu, jeśli występuje; dla modeli bez <think> zwraca tekst bez zmian."""
    text = str(text)
    # pełny blok <think>...</think> (także wielokrotny)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # samo domykające </think> bez otwarcia (reasoning ucięty na początku)
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()


def transcript_cues(prompt):
    tail = prompt.split("transcript to translate", 1)[-1]
    return n_cues(tail)


rows = []
for i in df[df["response"].notna()].index:
    resp = str(df.at[i, "response"])
    body = strip_think(resp)
    src_cues = transcript_cues(df.at[i, "prompt"])
    out_cues = n_cues(body)
    rows.append({
        "id": df.at[i, "id"],
        "lang": df.at[i, "target_lang"],
        "in_cue": src_cues,
        "out_cue": out_cues,
        "%": round(100 * out_cues / src_cues) if src_cues else 0,
        "ma_think": "</think>" in resp,
        "ma_SRT": bool(TS_RE.search(body)),
        "puste": len(body) == 0,
    })

check = pd.DataFrame(rows)
ok = (check["%"] >= 95).sum()
print(f"Kompletne (>=95% cue): {ok}/{len(check)}  |  mediana %: {int(check['%'].median()) if len(check) else 0}")
print(f"Puste odpowiedzi: {int(check['puste'].sum())}  |  bez SRT: {int((~check['ma_SRT']).sum())}")
check

In [ ]:
VERIFY_LANG = "pl"

sub = df[(df["target_lang"] == VERIFY_LANG) & df["response"].notna()]
if sub.empty:
    print(f"Brak odpowiedzi dla jezyka '{VERIFY_LANG}'")
else:
    row = sub.iloc[0]
    slug = row["slug"]
    reference = transcripts.loc[slug, f"{VERIFY_LANG}_timed"] if slug in transcripts.index else ""

    print("=" * 80)
    print(f"PROMPT ({row['id']})  - fragment")
    print("=" * 80)
    print(row["prompt"][:1200], "\n...[obciete]...\n")

    print("=" * 80)
    print(f"Oficjalne tlumaczenie TED ({VERIFY_LANG}), pierwsze cue")
    print("=" * 80)
    print("\n\n".join(str(reference).split("\n\n")[:6]))

    print("\n" + "=" * 80)
    print(f"Odpowiedz modelu ({MODEL_SHORT}), pierwsze cue (po usunieciu think)")
    print("=" * 80)
    print("\n\n".join(strip_think(row["response"]).split("\n\n")[:6]))